# RefMate Handball - OCR Processing

This notebook uses LightOnOCR-1B-1025 to convert cropped PDF images to text.

**Instructions:**
1. Runtime > Change runtime type > GPU (T4)
2. Upload your cropped images to `images/` folder
3. Run all cells
4. Download the generated `.txt` files

In [ ]:
# Install dependencies (need latest transformers for LightOnOcr classes)
!pip install -q git+https://github.com/huggingface/transformers accelerate torch pillow tqdm

In [ ]:
# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Create folders and upload images
import os
from google.colab import files

os.makedirs('images', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("Upload your cropped PNG images:")
uploaded = files.upload()

# Move uploaded files to images folder
for filename in uploaded.keys():
    os.rename(filename, f'images/{filename}')
    print(f"Moved: {filename}")

In [ ]:
# Alternative: Mount Google Drive
# Uncomment if you prefer to use Drive

# from google.colab import drive
# drive.mount('/content/drive')
# IMAGE_DIR = '/content/drive/MyDrive/refmate/images'
# OUTPUT_DIR = '/content/drive/MyDrive/refmate/output'

In [ ]:
# Load model
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

MODEL_ID = "lightonai/LightOnOCR-1B-1025"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

print("Loading model...")
processor = LightOnOcrProcessor.from_pretrained(MODEL_ID)
model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype
).to(device)
print("Model loaded!")

In [ ]:
# OCR function

def ocr_image(image_path):
    """Extract text from an image using LightOnOCR."""
    # Use URL/path format as expected by the model
    conversation = [
        {"role": "user", "content": [{"type": "image", "url": str(image_path)}]}
    ]

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    # Move tensors to device, casting only floating point tensors to dtype
    inputs = {
        k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    text = processor.decode(generated_ids, skip_special_tokens=True)

    return text.strip()

In [ ]:
# Process all images
from pathlib import Path
from datetime import date
from tqdm import tqdm
from transformers import AutoTokenizer

IMAGE_DIR = Path('images')
OUTPUT_DIR = Path('output')

images = sorted(IMAGE_DIR.glob('*.png'))
print(f"Found {len(images)} images to process")

pages_text = []
for image_path in tqdm(images, desc="Processing"):
    try:
        text = ocr_image(image_path.resolve())
        pages_text.append(f"<!-- Page: {image_path.stem} -->\n{text}")
        print(f"\u2713 {image_path.name}")
    except Exception as e:
        print(f"\u2717 {image_path.name}: {e}")
        pages_text.append(f"<!-- Page: {image_path.stem} - ERROR -->")

# Combine pages
combined_text = "\n\n---\n\n".join(pages_text)

# Build YAML front matter with token count
TOKENIZER_MODEL = "moonshotai/Kimi-K2.5"
print(f"Loading tokenizer: {TOKENIZER_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL, trust_remote_code=True)
token_count = len(tokenizer.encode(combined_text))

doc_name = IMAGE_DIR.name.replace("-", " ").title()
front_matter = (
    f"---\n"
    f"document: {doc_name}\n"
    f"source_file: {doc_name}.pdf\n"
    f"date_processed: {date.today().isoformat()}\n"
    f"pages: {len(pages_text)}\n"
    f"tokens: {token_count}\n"
    f"---\n\n"
)

# Save with front matter
output_file = OUTPUT_DIR / 'ocr_output.md'
output_file.write_text(front_matter + combined_text, encoding='utf-8')
print(f"\nSaved to {output_file} ({token_count} tokens)")

In [ ]:
# Download result
files.download('output/ocr_output.md')

In [ ]:
# Preview first 2000 characters
print(combined_text[:2000])